# Sample Superstore

## Environment Setup & Data Ingestion
We begin by loading essential data manipulation libraries (`pandas` and `numpy`) and reading the raw dataset. 

*Note: The file is ingested using `ISO-8859-1` encoding (Latin-1) to handle non-UTF-8 characters safely.*

## Dataset Diagnostics: Dimensions, Duplicates & Missingness
Before performing data transformations, we conduct a high level data integrity check to evaluate:
* Overall row and column count.
* Total number of duplicate entries across all features.
* Proportion of missing values ($NaNs$) per column.

In [2]:
import pandas as pd
import numpy as np

path="../data/raw/Sample-Superstore.csv"
df=pd.read_csv(path, encoding="ISO-8859-1")

# Dataset Dimension
print("="*10, "Dataset Dimension", "="*10)
row=df.shape[0]
col=df.shape[1]
print(f"Dataset Shape: There are {row} rows and {col} columns exist in dataset\n")

# Duplicate Values
print("="*10, "Duplicates", "="*10)
dupli=df.duplicated().sum()
print(f"Duplicates: There are {dupli} Duplicate values in dataset\n")

# Missing Values
print("="*10, "Missing Values", "="*10)
nulls=(df.isnull().sum()/len(df))*100
missing=nulls[nulls>0]
if not missing.empty:
    print("Missing Values(%)")
    print(missing)
else:
    print("Missing Values: NO Missing Values Found In Dataset")

========== Dataset Dimension ==========
Dataset Shape: There are 9994 rows and 21 columns exist in dataset

========== Duplicates ==========
Duplicates: There are 0 Duplicate values in dataset

========== Missing Values ==========
Missing Values: NO Missing Values Found In Dataset


In [3]:
print("="*10, "Dataset Description", "="*10, "\n")
df.info()

========== Dataset Description ========== 

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994

### Date Format Detection & Conversion

Date fields stored as text present high risks of silent parsing errors (e.g., misinterpreting `11/08/2016` as August 11 instead of November 8). 

To address this, we perform a deterministic audit on raw date strings:Evaluating whether position 1 or position 2 exceeds `12` proves whether the dataset follows `MM/DD/YYYY` or `DD/MM/YYYY`. 

Knowing the pattern allows us to explicitly pass `format='%m/%d/%Y'` into `pd.to_datetime()`, guaranteeing zero ambiguous date conversions.

In [4]:
print("="*10, "Date Column Inspection ", "="*10)
dates_split=df['Order Date'].astype(str).str.split('/',expand=True)

# Column split
date_first=df[dates_split[0].astype(float,errors='ignore')>12]
month_first=df[dates_split[1].astype(float,errors='ignore')>12]
print(f"Rows with 1st value > 12: {len(date_first)}")
print(f"Rows with 2nd value > 12: {len(month_first)}")

========== Date Column Inspection  ==========
Rows with 1st value > 12: 0
Rows with 2nd value > 12: 5952


In [5]:
# Date Conversion
df['Order Date']=pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']=pd.to_datetime(df['Ship Date'], format='%m/%d/%Y')

### Final Verification
Finally, we print `.info()` and display the top 5 records to verify that:
1. `Order Date` and `Ship Date` are cleanly cast to `datetime64[ns]`.
2. All non-null counts and memory consumption remain optimal.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   s

In [7]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Key Financial Metrics KPIs (Key Performance Indicator)

### High-Level Financial Performance (KPIs)

We summarize the core financial metrics across the entire dataset: **Total Revenue**, **Total Net Profit**, and the **Overall Profit Margin**.

In [8]:
sales=df['Sales'].sum()
profit=df['Profit'].sum()
margin=(profit/sales)
print("="*30, "Financial KPI", "="*30)
print(f"Total Sales: {sales:,.2f}")
print(f"Total Profit: {profit:,.2f}")
print(f"Profit Margin: {margin:.2%}")

============================== Financial KPI ==============================
Total Sales: 2,297,200.86
Total Profit: 286,397.02
Profit Margin: 12.47%


**Key Business Takeaway:** The business retains `$12.47` in profit for every `$100` generated in sales. While a double digit margin indicates baseline profitability, further breakdown by `Category, Sub Category, and Region` is required to identify underperforming segments and margin leakages.

### Category Level Performance

In [11]:
# Profit Margin By Category
category=df.groupby('Category').agg({'Sales':'sum','Profit':'sum'})
category['Profit Margin']=category['Profit']/category['Sales']
cat_sorted=category.sort_values(by='Sales',ascending=False)
cat_sorted.style.format({'Sales':'${:,.2f}','Profit':'${:,.2f}','Profit Margin':'{:.2%}'})

,Sales,Profit,Profit Margin
Category,,,
Technology,"$836,154.03","$145,454.95",17.40%
Furniture,"$741,999.80","$18,451.27",2.49%
Office Supplies,"$719,047.03","$122,490.80",17.04%


**Technology** is the strongest performer generating the highest revenue with a healthy `17.40% profit` margin. `Furniture` by contrast generates comparable revenue `(~$742K)` but converts almost none of it into profit its margin sits at just `2.49%`, the lowest of the three categories. This gap is the clearest red flag in the dataset

Furniture is pulling nearly a third of total sales while contributing barely any bottom line profit. The pattern is consistent with excessive discounting on this category, though this would need to be confirmed by looking at average discount rates before being treated as a settled cause. 

**Recommendation:** review discount policy on Furniture before continuing to push volume there.

### Regional Performance

In [10]:
# Profit Margin By Region
region=df.groupby('Region').agg({'Sales':'sum', 'Profit':'sum'})
region['Profit Margin']=region['Profit']/region['Sales']
reg_sorted=region.sort_values(by='Sales',ascending=False)
reg_sorted.style.format({'Sales':'${:,.2f}','Profit':'${:,.2f}','Profit Margin':'{:.2%}'})

,Sales,Profit,Profit Margin
Region,,,
West,"$725,457.82","$108,418.45",14.94%
East,"$678,781.24","$91,522.78",13.48%
Central,"$501,239.89","$39,706.36",7.92%
South,"$391,721.91","$46,749.43",11.93%


Among all four regions `West` leads in both total revenue `($725K)` and profitability `(14.94% margin)` making it the strongest performing region overall. 

**Central** is where the real concern lies: it generates more revenue than South `($501K vs $392K)` but converts far less of it into profit, a `7.92% margin` compared to `South's 11.93%`. In other words, Central is doing more sales volume while running less efficiently than a smaller region. This points to something specific happening in Central likely discounting or cost structure that's worth investigating before allocating further resources there.

## Top 5 Best Selling Products

In [ ]:
best_products=df.groupby(['Category','Sub-Category','Product Name']).agg({'Sales':'sum','Profit':'sum','Quantity':'sum'}).reset_index()
best_products['Profit Margin']=best_products['Profit']/best_products['Sales']
sorted_products=best_products.sort_values(by='Sales',ascending=False).head(5)
sorted_products.style.format({
    'Sales':'${:.2f}',
    'Profit':'${:.2f}',
    'Profit Margin':'{:.2%}'
})

,Category,Sub-Category,Product Name,Sales,Profit,Quantity,Profit Margin
1592,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,$61599.82,$25199.93,20,40.91%
716,Office Supplies,Binders,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,$27453.38,$7753.04,31,28.24%
1612,Technology,Machines,Cisco TelePresence System EX90 Videoconferencing Unit,$22638.48,$-1811.08,6,-8.00%
90,Furniture,Chairs,HON 5400 Series Task Chairs for Big and Tall,$21870.58,$0.00,39,0.00%
728,Office Supplies,Binders,GBC DocuBind TL300 Electric Binding System,$19823.48,$2233.51,37,11.27%


Among the top 5 best selling products by revenue, the `Canon imageCLASS 2200 Advanced Copier` leads with `$61.6K in sales`, generating `$25.2K in profit` at a healthy `40.91% margin`. The pattern breaks down for two other products in this list, however the `Cisco TelePresence System EX90 (-8% margin)` and the `HON 5400 Series Task Chairs (0.00% margin)` generate comparably high revenue but return little to no profit 

One is actively losing money the other generates 0 profit. it shows that high revenue products are generally expected to be reliable profit drivers, so having 2 of the top 5 best sellers fail to convert that revenue into profit suggests a pricing or cost issue specific to these SKUs.

# Top 5 Least Profitable Products

In [ ]:
less_selling__products=df.groupby(['Category','Sub-Category','Product Name']).agg({'Sales':'sum','Profit':'sum','Quantity':'sum'}).reset_index()
less_selling__products['Profit Margin']=less_selling__products['Profit']/less_selling__products['Sales']
sorted_products=less_selling__products.sort_values(by='Profit Margin',ascending=True).head(5)
sorted_products.style.format({
    'Sales':'${:.2f}',
    'Profit':'${:.2f}',
    'Profit Margin':'{:.2%}'
})

,Category,Sub-Category,Product Name,Sales,Profit,Quantity,Profit Margin
418,Office Supplies,Appliances,Eureka Disposable Bags for Sanitaire Vibra Groomer I Upright Vac,$1.62,$-4.47,2,-275.00%
13,Furniture,Bookcases,"Bush Westfield Collection Bookcases, Dark Cherry Finish, Fully Assembled",$90.88,$-190.85,3,-210.00%
425,Office Supplies,Appliances,Euro Pro Shark Stick Mini Vacuum,$170.74,$-325.63,11,-190.71%
1638,Technology,Machines,Okidata B401 Printer,$179.99,$-251.99,3,-140.00%
1658,Technology,Machines,Zebra GK420t Direct Thermal/Thermal Transfer Printer,$703.71,$-938.28,6,-133.33%


All five products in this list carry negative profit margins, meaning the company loses money on every unit sold. 

The worst case is the `Eureka Disposable Bags for Sanitaire Vibra Groomer` which sold for just `$1.62` but generated a `-$4.47 profit` a `-275% margin`, meaning the company effectively spends nearly `$3.75` for every **$1** it earns from this product, a loss of roughly `3x the selling price`. 

This pattern isn't isolated: it repeats across three different departments `Office Supplies, Furniture, and Technology` suggesting the issue isn't specific to one product line. That said, the financial exposure is limited: total losses stay under $1,000 per product, since sales volume for each is very low `(2–11 units)`. 

**Recommendation:** investigate pricing, discounting, or shipping costs on these specific SKUs before treating this as a wider systemic issue.

### Data Export
The cleaned dataset with audited date types, and verified missingness is exported to the processed data directory for further visualization and reporting.

In [ ]:
# Save cleaned CSV file in the desired location
df.to_csv('../data/processed/super-store-cleaned.csv', index=False)